# Question 1: Frequent Item Set Mining

In [1]:
import pandas as pd
from itertools import combinations
from collections import Counter

df = pd.read_csv('recipe_ingredient_table.csv')
transactions = [frozenset(g) for _, g in df.groupby('Recipe_ID')['Ingredient_Name']]
N = len(transactions)
print('Recipes (transactions):', N)

MIN_SUPPORT = 0.001
min_count = int(MIN_SUPPORT * N)
print('Minimum count =', min_count, 'recipes')

def apriori_upto3(transactions, min_count):
    c1 = Counter(i for t in transactions for i in t)
    L1 = {(i,): c for i, c in c1.items() if c >= min_count}
    freq_items = {i for (i,) in L1}

    c2 = Counter()
    for t in transactions:
        items = sorted(x for x in t if x in freq_items)
        c2.update(combinations(items, 2))
    L2 = {p: c for p, c in c2.items() if c >= min_count}

    L2_set = set(L2)
    c3 = Counter()
    for t in transactions:
        items = sorted(x for x in t if x in freq_items)
        items = [x for x in items]
        for tri in combinations(items, 3):
            a, b, c = tri
            if (a, b) in L2_set and (a, c) in L2_set and (b, c) in L2_set:  
                c3[tri] += 1
    L3 = {t: c for t, c in c3.items() if c >= min_count}
    return L1, L2, L3

L1, L2, L3 = apriori_upto3(transactions, min_count)
print('Frequent item sets --> size 1:', len(L1), '| size 2:', len(L2), '| size 3:', len(L3))

import os
OUT_DIR = 'outputs/Q1'
os.makedirs(OUT_DIR, exist_ok=True)

Recipes (transactions): 10168
Minimum count = 10 recipes
Frequent item sets --> size 1: 1377 | size 2: 16252 | size 3: 70978


## Q1a : Complete lists of item sets of size 1, 2 and 3, saved as CSV

In [2]:
def to_df(L, k):
    out = pd.DataFrame({'Itemset': [' | '.join(s) for s in L],
                        'Count': list(L.values())})
    out['Support'] = out['Count'] / N
    out['Size'] = k
    return out.sort_values(['Count', 'Itemset'], ascending=[False, True]).reset_index(drop=True)

df1, df2, df3 = to_df(L1, 1), to_df(L2, 2), to_df(L3, 3)
df1.to_csv(os.path.join(OUT_DIR, 'Q1a_itemsets_size1.csv'), index=False)
df2.to_csv(os.path.join(OUT_DIR, 'Q1a_itemsets_size2.csv'), index=False)
df3.to_csv(os.path.join(OUT_DIR, 'Q1a_itemsets_size3.csv'), index=False)

print('Length of itemsets size 1:', len(df1))
print('Length of itemsets size 2:', len(df2))
print('Length of itemsets size 3:', len(df3), '\n')

print('Output files saved in output/Q1/: Q1a_itemsets_size1.csv, Q1a_itemsets_size2.csv, Q1a_itemsets_size3.csv')

Length of itemsets size 1: 1377
Length of itemsets size 2: 16252
Length of itemsets size 3: 70978 

Output files saved in output/Q1/: Q1a_itemsets_size1.csv, Q1a_itemsets_size2.csv, Q1a_itemsets_size3.csv


## Q1b : Top 20 item sets of size 1, 2 and 3

In [3]:
top1, top2, top3 = df1.head(20), df2.head(20), df3.head(20)
top1.to_csv(os.path.join(OUT_DIR, 'Q1b_top20_size1.csv'), index=False)
top2.to_csv(os.path.join(OUT_DIR, 'Q1b_top20_size2.csv'), index=False)
top3.to_csv(os.path.join(OUT_DIR, 'Q1b_top20_size3.csv'), index=False)

In [4]:
print('Output file saved in output/Q1/Q1b_top20_size1.csv')
print('Top 20 item sets of size 1'); display(top1)

Output file saved in output/Q1/Q1b_top20_size1.csv
Top 20 item sets of size 1


,Itemset,Count,Support,Size
0,salt,5339,0.525079,1
1,water,3279,0.322482,1
2,oil,3007,0.295732,1
3,garlic,1840,0.180960,1
4,ginger,1651,0.162372,1
5,onion,1640,0.161290,1
6,ghee,1511,0.148603,1
7,sugar,1408,0.138474,1
8,cumin seeds,1220,0.119984,1
9,turmeric powder,1165,0.114575,1


In [5]:
print('Output file saved in output/Q1/Q1b_top20_size2.csv')
print('Top 20 item sets of size 2'); display(top2)

Output file saved in output/Q1/Q1b_top20_size2.csv
Top 20 item sets of size 2


,Itemset,Count,Support,Size
0,oil | salt,2807,0.276062,2
1,salt | water,2585,0.254229,2
2,oil | water,1714,0.168568,2
3,garlic | salt,1679,0.165126,2
4,onion | salt,1545,0.151947,2
5,ginger | salt,1487,0.146243,2
6,cumin seeds | salt,1062,0.104445,2
7,salt | turmeric powder,1053,0.103560,2
8,ginger | water,1006,0.098938,2
9,curry leaves | salt,991,0.097463,2


In [6]:
print('Output file saved in output/Q1/Q1b_top20_size3.csv')
print('Top 20 item sets of size 3'); display(top3)

Output file saved in output/Q1/Q1b_top20_size3.csv
Top 20 item sets of size 3


,Itemset,Count,Support,Size
0,oil | salt | water,1616,0.158930,3
1,ginger | salt | water,928,0.091267,3
2,oil | onion | salt,915,0.089988,3
3,ginger | oil | salt,904,0.088906,3
4,garlic | salt | water,880,0.086546,3
5,onion | salt | water,876,0.086153,3
6,garlic | ginger | salt,849,0.083497,3
7,garlic | oil | salt,815,0.080153,3
8,curry leaves | oil | salt,707,0.069532,3
9,garlic | onion | salt,664,0.065303,3


## Q1c: Support and Confidence in Item Set Mining

**Support**
- Support of an item set X is the fraction of transactions (recipes) that contain **all** items of X:  
  `support(X) = (number of recipes containing X) / (total number of recipes)`.
- It measures how *frequent/popular* an item set is. An item set is called **frequent** if its support ≥ a minimum support threshold. Example: `support({salt, oil})` = share of recipes using both salt and oil.
- Support is anti-monotone: adding items to a set can never increase its support. Apriori uses this to prune the search.

**Confidence**
- Confidence of a rule X → Y is the conditional probability of finding Y in a recipe given that X is present:  
  `confidence(X → Y) = support(X ∪ Y) / support(X)`.
- It measures how *reliable* the rule is. A value of 1 means every recipe containing X also contains Y; for example `confidence(onion → salt)` is the fraction of onion-containing recipes that also contain salt.
- Unlike support, confidence is directional (X → Y ≠ Y → X) and does not account for how common Y is on its own (that is what *lift* corrects).